In [171]:
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import math

In [213]:
sol = scipy.io.loadmat('./Solutions/alpha_1_5000_eta_10_800.mat')

In [214]:
sol_struct = sol['sol']
maxCl = sol_struct[0,0]

In [215]:
#print(maxCl['belongTo'])
maxCl['belongTo'] = np.squeeze(maxCl['belongTo'])
print(maxCl['belongTo'])

[3 3 3 ... 9 3 3]


In [216]:
print(maxCl['weight'])

[[-0.54699463  0.59787794 -0.4493696  -0.54928048  0.20089362 -0.44658843
  -0.21487904 -0.32250666  0.90784895 -0.80988539  0.75603758]
 [-0.94663083  0.40282651 -0.19490694 -0.07355789  0.02266339  0.11995964
   0.54333093  0.52727485  0.8807908   0.37945782  0.39154315]
 [-0.27877732  0.12271832 -0.14349068  0.63016638  0.41589398  0.60712128
  -0.19010258  0.70277687  0.96157334  0.607764   -0.94516391]
 [-0.22630764  0.37252508  0.35515764 -0.55953285  0.08925681 -0.88214324
  -0.64074043  0.73618284  0.19868467  0.12600709 -0.87084257]
 [ 0.28502427  0.58767926  0.26734643 -0.20385731  0.33438669 -0.81949451
  -0.53638667  0.85359599  0.01538425  0.06605314 -0.03039008]
 [ 0.27583298  0.71164868  0.20389105 -0.66239586  0.57155947 -0.75049923
  -0.97232026  0.78585421 -0.00583522 -0.57961908 -0.45835575]]


In [217]:
reward_func = maxCl['weight']
reward_reshape = reward_func.T
print(reward_reshape)

[[-0.54699463 -0.94663083 -0.27877732 -0.22630764  0.28502427  0.27583298]
 [ 0.59787794  0.40282651  0.12271832  0.37252508  0.58767926  0.71164868]
 [-0.4493696  -0.19490694 -0.14349068  0.35515764  0.26734643  0.20389105]
 [-0.54928048 -0.07355789  0.63016638 -0.55953285 -0.20385731 -0.66239586]
 [ 0.20089362  0.02266339  0.41589398  0.08925681  0.33438669  0.57155947]
 [-0.44658843  0.11995964  0.60712128 -0.88214324 -0.81949451 -0.75049923]
 [-0.21487904  0.54333093 -0.19010258 -0.64074043 -0.53638667 -0.97232026]
 [-0.32250666  0.52727485  0.70277687  0.73618284  0.85359599  0.78585421]
 [ 0.90784895  0.8807908   0.96157334  0.19868467  0.01538425 -0.00583522]
 [-0.80988539  0.37945782  0.607764    0.12600709  0.06605314 -0.57961908]
 [ 0.75603758  0.39154315 -0.94516391 -0.87084257 -0.03039008 -0.45835575]]


In [218]:
policies = maxCl['policy']
policies = policies.T
np.shape(policies)

(11, 750)

In [25]:
mimic = pd.read_csv("./dataSets/Sepsis_data_process.csv")
demo_states = np.load("./dataSets/demonstrations_states_sample.npy")
demo_actions = np.load("./dataSets/demonstrations_actions_sample.npy")


In [26]:
tupleToNumericStates = {}
numericStatesToTuple = {}
count = 0
for i in range(25):
    for j in range(30):
        tupleToNumericStates[(i,j)] = count
        numericStatesToTuple[count] = (i,j)
        count += 1


In [27]:
print(demo_actions[5])
print(demo_states[5])

[ 62. 303. 182. 212. 212. 212. 182. 183. 151. 151. 181. 181. 181. 211.
 181. 181. 181. 183. 211. 151. 131.]
[ 62. 303. 182. 212. 212. 212. 182. 183. 151. 151. 181. 181. 181. 211.
 181. 181. 181. 183. 211. 151. 131.]


In [219]:
Qcells = maxCl['Qcells']

In [29]:
# recreate the original mimic table
def recreateOriginalTable(demonstrations, learnt_policy):
    icuids = list(map(lambda x: x[20], demonstrations))
    
    mimic_new = mimic[mimic['icustayid'].isin(icuids)]
    
    return mimic_new, len(icuids)


In [30]:

# def mapActions(mimic_table, learnt_policy):
#     actions = []
#     for i in range(len(mimic_table)):
#         curr_state = mimic_table.iloc[i]['discrete_states']
#         action_for_state = learnt_policy[int(curr_state)]
#         action_for_state -= 1
#         actions.append(action_for_state)
#     mimic_table['piL'] = actions

In [31]:
# Compute the probability distribution of actions for each state
action_distr = {}
for i in range(len(mimic)):
    state = mimic.iloc[i]['discrete_states']
    action = mimic.iloc[i]['actions']
    
    if state not in action_distr:
        action_distr[state] = [0 for i in range(25)]
    
    action_distr[state][int(action)] += 1

# normalise action distribution values
for state in action_distr:
    tot = np.sum(action_distr[state])
    vals = [x/tot for x in action_distr[state]]
    action_distr[state] = vals

In [245]:
class ClusterObject:
    def __init__(self, index, size, table, reward_function, Q_vals):
        self.no_trajs = size
        self.clusterId = index
        self.table = table
        self.reward = reward_function
        self.feature_count = self.calculateRewards()
        self.Q_vals = Q_vals
        self.action_distr = self.computeActionDistr()
        
        self.policyMatch = self.calculatePolicyMatching()
        self.statistics_sofa, self.statistics_AL = self.statistics()
        
    def calculatePolicyMatching(self):
        states = []
        tot = len(self.table)
        matches = 0
        for i in range(len(self.table)):
            state = int(self.table.iloc[i]['discrete_states'])
            actionsE = self.table.iloc[i]['actions']
            states.append(state)
        states = np.unique(states)
        
        state_dct = {}
        
        for state in states:
            # log likelihood values of Q
            Q_s_a = self.Q_vals[state]

            # how many top 3 Q values match with expert top 3 action
            state_dct[state] = 0

            # get top 3 log likelihoods for learnt policy
            log_likelihoods_and_actions_L = sorted( [(x,i) for (i,x) in enumerate(Q_s_a)], reverse=True )[:3]
            # find top 3 log likelihoods for experts action at that state
            action_distrs = self.action_distr[state]

            # top 3 actions
            probs_and_actions = sorted( [(x,i) for (i,x) in enumerate(action_distrs)], reverse=True )[:3]
            for prob, action in probs_and_actions:
                # if any of top 3 action appears in top 3 likelihoods of the learnt LLH policy
                for Q_val, actionL in log_likelihoods_and_actions_L:
                    if action == actionL:
                        state_dct[state] += 1
                                                
        no_states = len(state_dct)
        matches = 0
                
        for key in state_dct:
            if state_dct[key] != 0:
                matches += 1
                
        return matches / no_states * 100
    
            
    
#     # Compute the probability distribution of actions for each state
    def computeActionDistr(self):
        action_distr = {}
        for i in range(len(self.table)):
            state = self.table.iloc[i]['discrete_states']
            action = self.table.iloc[i]['actions']

            if state not in action_distr:
                action_distr[state] = [0 for i in range(25)]

            action_distr[state][int(action)] += 1
            
        # chi squared test for dictionary of states to frequency list of actions taken at that state.
        chi_squared = action_distr.copy()
        self.chi_squared = chi_squared
        
    
        # normalise action distribution values
        for state in action_distr:
            tot = np.sum(action_distr[state])
            vals = [x/tot for x in action_distr[state]]
            action_distr[state] = vals

        return action_distr
    
    def featureToNumber(self, feature_vect):
        # SOFA increase
        if feature_vect[0] == 1:
            # lactate increase
            if feature_vect[3] == 1:
                return 1
            # lactate maintain
            if feature_vect[4] == 1:
                return 2
            # lactate decrease
            if feature_vect[5] == 1:
                return 3
        # SOFA maintain
        elif feature_vect[1] == 1:
            # lactate drop
            if feature_vect[3] == 1:
                return 4
            # lactate maintain
            if feature_vect[4] == 1:
                return 5
            # lactate drop
            if feature_vect[5] == 1:
                return 6
        # SOFA drop
        else:
            # lact inc
            if feature_vect[3] == 1:
                return 7
            # lact maint
            if feature_vect[4] == 1:
                return 8
            # lact drop
            if feature_vect[5] == 1:
                return 9

            
    def calculateRewards(self):
        rewards = []
        feature_count = {}
        for i in range(len(self.table) - 1):
            feature_vector = [0 for i in range(6)]
            cur_sofa = self.table.iloc[i]['SOFA']
            next_sofa = self.table.iloc[i+1]['SOFA']
            
            cur_lact = self.table.iloc[i]['Arterial_lactate']
            next_lact = self.table.iloc[i+1]['Arterial_lactate']
            
            cur_lact = math.floor(cur_lact)
            next_lact = math.floor(next_lact)
            
            if cur_sofa > next_sofa:
                feature_vector[2] = 1
            if cur_sofa == next_sofa:
                feature_vector[1] = 1
            if cur_sofa < next_sofa:
                feature_vector[0] = 1
                
            if cur_lact > next_lact:
                feature_vector[5] = 1
            if cur_lact == next_lact:
                feature_vector[4] = 1
            if cur_lact < next_lact:
                feature_vector[3] = 1
            
            hashed = self.featureToNumber(feature_vector)
            if hashed not in feature_count:
                feature_count[hashed] = 0
            feature_count[hashed] += 1
            
            reward = np.dot(feature_vector, self.reward.T)
            rewards.append(reward)
            
        rewards.append(-1)
        self.table['reward'] = rewards
        
        return feature_count

        
    def computeStats(self, variable_list):
        # compute mean, stdev of patients belonging to the different clusters
        statistics = {}
        mean_sofa = 0
        mean_al = 0
        icuids = np.unique(self.table['icustayid'])
        mort = 0
        for ids in icuids:
            died = self.table.loc[self.table['icustayid'] == ids].iloc[0]['died_in_hosp']
            if (died):
                mort += 1
        
        ave_max_sofa = []
        ave_max_al = []
        
        for ids in icuids:
            # compute average
            init_sofa = self.table.loc[self.table['icustayid'] == ids].iloc[0]['SOFA']
            init_al = self.table.loc[self.table['icustayid'] == ids].iloc[0]['Arterial_lactate']
            
            mean_sofa += init_sofa
            mean_al += init_al
            
            max_sofa = self.table.loc[self.table['icustayid'] == ids]['SOFA'].max()
            max_al = self.table.loc[self.table['icustayid'] == ids]['Arterial_lactate'].max()
            
            ave_max_sofa.append(max_sofa)
            ave_max_al.append(max_al)
            
            
        mean_sofa = mean_sofa / len(icuids)
        mean_al = mean_al / len(icuids)
        
        self.ave_max_sofa = np.mean(ave_max_sofa)
        self.ave_max_al = np.mean(ave_max_al)
        
        self.init_sofa = mean_sofa
        self.mean_al = mean_al
                
        statistics['died_in_hosp']= mort / len(icuids)
        
        for variable in variable_list:
            df_vars = self.table[variable].to_list()
            df_var_mean = self.table[variable].mean()
            df_var_std = self.table[variable].std()
            
            print(variable + ":")
            print("Mean score: ", df_var_mean)
            print("Stddev: ", df_var_std)
            # store mean, stddev and the whole series
            statistics[variable] = [df_var_mean, df_var_std, df_vars]
    
    
        print("Mortality: ", statistics['died_in_hosp'] * 100)
        return statistics
        # initial SOFA
        #for i in range(len(self.table)):
        
    def statistics(self):
        #. count number of times SOFA inc, maintain, dec
        statistics_sofa = [0 for i in range(3)]
        statistics_AL = [0 for i in range(3)]
        for i in range(len(self.table) - 1):
            curr_sofa = self.table.iloc[i]['SOFA']
            next_sofa = self.table.iloc[i+1]['SOFA']
            
            curr_lactate = self.table.iloc[i]['Arterial_lactate']
            next_lactate = self.table.iloc[i+1]['Arterial_lactate']
            
            curr_lactate = math.floor(curr_lactate)
            next_lactate = math.floor(next_lactate)
            
            if curr_sofa > next_sofa:
                statistics_sofa[2] += 1
            if curr_sofa == next_sofa:
                statistics_sofa[1] += 1
            if curr_sofa < next_sofa:
                statistics_sofa[0] += 1
                
            if curr_lactate > next_lactate:
                statistics_AL[2] += 1
            if curr_lactate == next_lactate:
                statistics_AL[1] += 1
            if curr_lactate < next_lactate:
                statistics_AL[0] += 1

        return [x / sum(statistics_sofa) for x in statistics_sofa], [x / sum(statistics_AL) for x in statistics_AL]
    
        

In [246]:
def createClusterDictionary(maxCl, demonstrations, rewards, piL, Qcells):
    dct = {}
    belongTo = maxCl['belongTo']
    
    for i in range(1, int(max(belongTo)) + 1):
        indices = np.where(belongTo == i) # indexs of cluster assignemnt = i 
        trajectory_clusters = demonstrations[indices]
        
        # 750x1 vector
        cluster_policy = piL[i - 1]
        
        # 6x1 vector
        cluster_reward = rewards[i - 1]
        
        cluster_table, size = recreateOriginalTable(trajectory_clusters, cluster_policy)
        
        Q_vals = Qcells[i - 1][0]

        # append policies
        
        #print(len(cluster_table))
        dct[i] = ClusterObject(i, size, cluster_table, cluster_reward, Q_vals)
    
    return dct

In [247]:
clusterDct = createClusterDictionary(maxCl, demo_states, reward_reshape, policies, Qcells)

/Users/ongyichong/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:155: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [223]:
# print policy matchings
for clustObj in clusterDct.values():
    print("Cluster id: " + str(clustObj.clusterId))
    print("Cluster size: " + str(clustObj.no_trajs))
    print("Policy match: " + str(clustObj.policyMatch))
    print("Cluster weight: " + str(clustObj.reward))
    print("=========================================")

Cluster id: 1
Cluster size: 6
Policy match: 44.0
Cluster weight: [-0.54699463 -0.94663083 -0.27877732 -0.22630764  0.28502427  0.27583298]
Cluster id: 2
Cluster size: 216
Policy match: 58.139534883720934
Cluster weight: [0.59787794 0.40282651 0.12271832 0.37252508 0.58767926 0.71164868]
Cluster id: 3
Cluster size: 3625
Policy match: 68.01007556675063
Cluster weight: [-0.4493696  -0.19490694 -0.14349068  0.35515764  0.26734643  0.20389105]
Cluster id: 4
Cluster size: 127
Policy match: 41.81818181818181
Cluster weight: [-0.54928048 -0.07355789  0.63016638 -0.55953285 -0.20385731 -0.66239586]
Cluster id: 5
Cluster size: 9
Policy match: 31.25
Cluster weight: [0.20089362 0.02266339 0.41589398 0.08925681 0.33438669 0.57155947]
Cluster id: 6
Cluster size: 69
Policy match: 52.5
Cluster weight: [-0.44658843  0.11995964  0.60712128 -0.88214324 -0.81949451 -0.75049923]
Cluster id: 7
Cluster size: 1
Policy match: 50.0
Cluster weight: [-0.21487904  0.54333093 -0.19010258 -0.64074043 -0.53638667 -0.

In [43]:
# # list of cluster objects
# def combineClusters(mimic, maxCl, demonstrations, rewards, piL, Qcells):
#     indices = None
#     for clust in clusterobjs:
#         indexs = np.where(clust.clusterId)
#         if indices is None:
#             indices = indexs
#         else:
#             indices = np.concatenate((indices, indexs))
    
    
# belongTo = maxCl['belongTo']

# # cluster 2 and cluster 3 combine
# indices_2 = np.where(belongTo == 2)
# indices_3 = np.where(belongTo == 3)
# indices_9 = np.where(belongTo == 9)
# indices = np.concatenate([indices_3[0], indices_2[0], indices_9[0]])

# trajectory_clusters = demo_states[indices]

# # cluster 2 weights 
# cluster_policy = policies[2]

# # 6x1 vector
# cluster_reward = reward_reshape[2]

# cluster_table, size = recreateOriginalTable(trajectory_clusters, cluster_policy)

# Q_vals = Qcells[2][0]

# # append policies


# combined_clust = ClusterObject(i, size, cluster_table, cluster_reward, Q_vals)
# print(combined_clust.policyMatch)

In [95]:
# what can we do to analyse the different trajectory clustering ?
# What could be the cause of the arising of the different reward functions ?
# compare policy distributions by Chi squared
from scipy.stats import chisquare

def compare_action_distributions(action_distr_1, action_distr_2):
    # assume cluster 3 is the population distribution     
    freq_1 = [0 for i in range(25)]
    freq_2 = [0 for i in range(25)]
    tot = 0
    count = 0
    # get the frequency distribution
    for state in action_distr_1:
        # check if the state exist in action_distr_2
        if state in action_distr_2:
            tot += 1
            # let freq_2 be the population and action_distr_1 being sampled from it 
            freq_1 = np.add(freq_1, action_distr_1[state])
            freq_2 = np.add(freq_2, action_distr_2[state])            
            
            # if statistically significant at 10% level
            #if p_val < 0.1:
                #count += 1
                
            # check for the numbers for each action has to be > 5. Get index of actions without numbers = 5
            new_freq_1 = []
            new_freq_2 = []
            for action_1,action_2 in zip(freq_1, freq_2):
                if action_1 < 5 or action_2 < 5:
                    continue
                new_freq_1.append(action_1)
                new_freq_2.append(action_2)



            n_2 = sum(new_freq_2)
            n_1 = sum(new_freq_1)

            # compute p_i 
            p_i_list = []
            for i in range(len(new_freq_2)):
                p_i = new_freq_2[i] / n_2 # probability of belonging to this category
                p_i_list.append(p_i)

            f_exp = []
            # compute expected frequency
            for i in range(len(new_freq_1)):
                expected_freq = int(p_i_list[i] * n_1)
                f_exp.append(expected_freq)

            statistic, p_val = chisquare(new_freq_1, f_exp = f_exp, ddof = 2)

            if p_val < 0.05:
                count += 1

    return count / tot
    
    
    # number of distributions for each state that are different
                
chi_2 = compare_action_distributions(clusterDct[2].chi_squared, clusterDct[3].chi_squared)
chi_9 = compare_action_distributions(clusterDct[9].chi_squared, clusterDct[3].chi_squared)
chi_4 = compare_action_distributions(clusterDct[8].chi_squared, clusterDct[3].chi_squared)

print(chi_2)
print(chi_9)
print(chi_4)

1.0
1.0
0.9183673469387755


In [102]:
with open('../data/state_features.txt') as f:
    state_features = f.read().split()

In [103]:
print(state_features)  

['Albumin', 'Arterial_BE', 'Arterial_lactate', 'Arterial_pH', 'BUN', 'CO2_mEqL', 'Calcium', 'Chloride', 'Creatinine', 'DiaBP', 'FiO2_1', 'GCS', 'Glucose', 'HCO3', 'HR', 'Hb', 'INR', 'Ionised_Ca', 'Magnesium', 'MeanBP', 'PT', 'PTT', 'PaO2_FiO2', 'Platelets_count', 'Potassium', 'RR', 'SGOT', 'SGPT', 'SIRS', 'SOFA', 'Shock_Index', 'Sodium', 'SpO2', 'SysBP', 'Temp_C', 'Total_bili', 'WBC_count', 'Weight_kg', 'age', 'elixhauser', 'gender', 'mechvent', 'output_4hourly', 'output_total', 'paCO2', 'paO2', 're_admission', 'bloc']


{0: 0,
 1: 0,
 2: 0,
 3: 0,
 4: 1,
 5: 2,
 6: 2,
 9: 1,
 11: 2,
 30: 3,
 31: 2,
 32: 2,
 33: 3,
 34: 2,
 35: 2,
 36: 2,
 38: 3,
 39: 1,
 40: 3,
 42: 3,
 44: 3,
 60: 2,
 61: 1,
 62: 2,
 63: 3,
 64: 2,
 65: 2,
 66: 2,
 67: 2,
 70: 2,
 73: 2,
 75: 3,
 90: 2,
 91: 0,
 92: 0,
 93: 0,
 94: 0,
 95: 2,
 96: 1,
 99: 1,
 102: 2,
 105: 3,
 108: 3,
 120: 0,
 121: 0,
 122: 0,
 123: 0,
 124: 0,
 125: 1,
 126: 1,
 127: 1,
 128: 1,
 129: 1,
 130: 3,
 131: 1,
 132: 1,
 136: 3,
 143: 3,
 150: 0,
 151: 0,
 152: 0,
 153: 0,
 154: 0,
 155: 1,
 156: 2,
 157: 0,
 158: 3,
 159: 1,
 160: 2,
 162: 2,
 164: 2,
 169: 3,
 180: 0,
 181: 2,
 182: 0,
 183: 0,
 184: 0,
 185: 0,
 186: 0,
 187: 2,
 188: 2,
 189: 1,
 192: 1,
 193: 1,
 198: 3,
 203: 3,
 210: 0,
 211: 1,
 212: 0,
 213: 0,
 214: 0,
 215: 0,
 216: 0,
 217: 0,
 218: 0,
 219: 0,
 220: 0,
 222: 2,
 224: 2,
 240: 0,
 241: 1,
 242: 1,
 243: 0,
 244: 0,
 245: 0,
 246: 0,
 247: 0,
 248: 0,
 249: 0,
 270: 0,
 271: 1,
 272: 1,
 273: 2,
 274: 0,
 275: 0,
 276: 1,
 277

In [137]:
stats = clusterDct[3].computeStats(state_features)

Albumin:
Mean score:  2.9189128960924897
Stddev:  0.6618623160845492
Arterial_BE:
Mean score:  0.935048369037101
Stddev:  4.803099918174959
Arterial_lactate:
Mean score:  1.979958362295384
Stddev:  1.6062953126012445
Arterial_pH:
Mean score:  7.4001829977622435
Stddev:  0.06980784192008885
BUN:
Mean score:  32.81952840224158
Stddev:  24.513039545962652
CO2_mEqL:
Mean score:  26.552533952419132
Stddev:  5.565005311071293
Calcium:
Mean score:  8.31730579807841
Stddev:  0.7656411724629923
Chloride:
Mean score:  104.87176154802395
Stddev:  6.3277584635887685
Creatinine:
Mean score:  1.4963413170144135
Stddev:  2.1205765174898463
DiaBP:
Mean score:  57.53437338740257
Stddev:  13.144023403928912
FiO2_1:
Mean score:  0.46886233413075207
Stddev:  0.1608415977225029
GCS:
Mean score:  11.662788382752089
Stddev:  3.6088670531006577
Glucose:
Mean score:  140.80046072562843
Stddev:  46.0343069959419
HCO3:
Mean score:  25.160447548420183
Stddev:  5.128990112511372
HR:
Mean score:  87.92667692960536


In [248]:
#stats = clusterDct[2].computeStats(state_features)
print(clusterDct[2].feature_count)
print(clusterDct[2].reward)
print(clusterDct[2].no_trajs)

# reward ranking
reward = clusterDct[2].reward
print("reward 1: ", reward[0] + reward[3])
print("reward 2: ", reward[0] + reward[4])
print("reward 3: ", reward[0] + reward[5])
print("reward 4: ", reward[1] + reward[3])
print("reward 5: ", reward[1] + reward[4])
print("reward 6: ", reward[1] + reward[5])
print("reward 7: ", reward[2] + reward[3])
print("reward 8: ", reward[2] + reward[4])
print("reward 9: ", reward[2] + reward[5])

print(clusterDct[2].computeStats(state_features))

print("mean SOFA: ", clusterDct[2].init_sofa)
print("mean AL: ", clusterDct[2].mean_al)

print("max SOFA: ", clusterDct[2].ave_max_sofa)
print("max AL: ", clusterDct[2].ave_max_al)

#policies[1] - 1

{7: 272, 8: 799, 5: 1086, 4: 207, 6: 189, 2: 752, 3: 305, 9: 306, 1: 307}
[0.59787794 0.40282651 0.12271832 0.37252508 0.58767926 0.71164868]
216
reward 1:  0.9704030248832844
reward 2:  1.1855572046109495
reward 3:  1.3095266196030093
reward 4:  0.7753515907361437
reward 5:  0.9905057704638087
reward 6:  1.1144751854558685
reward 7:  0.4952434057036177
reward 8:  0.7103975854312827
reward 9:  0.8343670004233426
Albumin:
Mean score:  3.0195632257714786
Stddev:  0.6806902898690299
Arterial_BE:
Mean score:  0.9861143002230893
Stddev:  4.8329461560638185
Arterial_lactate:
Mean score:  1.8666725298784568
Stddev:  1.2830552147178793
Arterial_pH:
Mean score:  7.401648241127666
Stddev:  0.0707892842883702
BUN:
Mean score:  22.805785292143092
Stddev:  18.41890032258119
CO2_mEqL:
Mean score:  26.2480560789656
Stddev:  5.204835412462337
Calcium:
Mean score:  8.35379231982121
Stddev:  0.688326399106809
Chloride:
Mean score:  104.60266065956485
Stddev:  6.452040026427988
Creatinine:
Mean score:  1

In [254]:
#stats = clusterDct[9].computeStats(state_features)
print(clusterDct[9].feature_count)
print(clusterDct[9].reward)

reward = clusterDct[9].reward
print("reward 1: ", reward[0] + reward[3])
print("reward 2: ", reward[0] + reward[4])
print("reward 3: ", reward[0] + reward[5])
print("reward 4: ", reward[1] + reward[3])
print("reward 5: ", reward[1] + reward[4])
print("reward 6: ", reward[1] + reward[5])
print("reward 7: ", reward[2] + reward[3])
print("reward 8: ", reward[2] + reward[4])
print("reward 9: ", reward[2] + reward[5])

clusterDct[9].computeStats(state_features)

print("mean SOFA: ", clusterDct[9].init_sofa)
print("mean AL: ", clusterDct[9].mean_al)


print("max SOFA: ", clusterDct[9].ave_max_sofa)
print("max AL: ", clusterDct[9].ave_max_al)


{2: 2898, 4: 807, 3: 1425, 7: 1354, 1: 1482, 8: 3140, 9: 1495, 5: 3292, 6: 772}
[ 0.90784895  0.8807908   0.96157334  0.19868467  0.01538425 -0.00583522]
reward 1:  1.1065336123398049
reward 2:  0.9232331991485248
reward 3:  0.9020137276975568
reward 4:  1.0794754664529174
reward 5:  0.8961750532616373
reward 6:  0.8749555818106693
reward 7:  1.1602580036173693
reward 8:  0.9769575904260892
reward 9:  0.9557381189751213
Albumin:
Mean score:  2.9608684078875673
Stddev:  0.6639654347796928
Arterial_BE:
Mean score:  0.6977167472621384
Stddev:  4.985801747936526
Arterial_lactate:
Mean score:  2.0469325693583973
Stddev:  1.6711941670483141
Arterial_pH:
Mean score:  7.397904429509236
Stddev:  0.07145021148167142
BUN:
Mean score:  28.976968672678215
Stddev:  22.455457194791933
CO2_mEqL:
Mean score:  26.384395237349143
Stddev:  5.598269937809237
Calcium:
Mean score:  8.357196638136879
Stddev:  0.7912357873702306
Chloride:
Mean score:  104.3331529909329
Stddev:  6.360406263427561
Creatinine:
Me

In [252]:
#stats = clusterDct[4].computeStats(state_features)
print(clusterDct[3].feature_count)
print(clusterDct[3].reward)
print(clusterDct[3].statistics_sofa)
print(clusterDct[3].statistics_AL)

reward = clusterDct[3].reward
print("reward 1: ", reward[0] + reward[3])
print("reward 2: ", reward[0] + reward[4])
print("reward 3: ", reward[0] + reward[5])
print("reward 4: ", reward[1] + reward[3])
print("reward 5: ", reward[1] + reward[4])
print("reward 6: ", reward[1] + reward[5])
print("reward 7: ", reward[2] + reward[3])
print("reward 8: ", reward[2] + reward[4])
print("reward 9: ", reward[2] + reward[5])

clusterDct[3].computeStats(state_features)

print("mean SOFA: ", clusterDct[3].init_sofa)
print("mean AL: ", clusterDct[3].mean_al)

print("max SOFA: ", clusterDct[3].ave_max_sofa)
print("max AL: ", clusterDct[3].ave_max_al)

{8: 13190, 5: 15254, 2: 12651, 1: 6015, 7: 5521, 3: 5542, 9: 6056, 6: 3578, 4: 3460}
[-0.4493696  -0.19490694 -0.14349068  0.35515764  0.26734643  0.20389105]
[0.3396803569674604, 0.312795543519441, 0.34752409951309865]
[0.21041996997207685, 0.5766343468926712, 0.21294568313525195]
reward 1:  -0.09421195367335478
reward 2:  -0.18202317125946954
reward 3:  -0.24547854867826735
reward 4:  0.1602507021508759
reward 5:  0.07243948456476113
reward 6:  0.008984107145963316
reward 7:  0.21166696349577108
reward 8:  0.12385574590965631
reward 9:  0.0604003684908585
Albumin:
Mean score:  2.917424594698603
Stddev:  0.6606248265920549
Arterial_BE:
Mean score:  0.9658194073123495
Stddev:  4.810398598691684
Arterial_lactate:
Mean score:  1.9709491122610248
Stddev:  1.6177926936645441
Arterial_pH:
Mean score:  7.40027735499717
Stddev:  0.0700317382935842
BUN:
Mean score:  33.197786775905016
Stddev:  24.714197383813616
CO2_mEqL:
Mean score:  26.56480115128707
Stddev:  5.61359846568175
Calcium:
Mean s

In [155]:
#stats = clusterDct[6].computeStats(state_features)
print(clusterDct[6].feature_count)
print(clusterDct[6].reward)

reward = clusterDct[6].reward
print("reward 1: ", reward[0] + reward[3])
print("reward 2: ", reward[0] + reward[4])
print("reward 3: ", reward[0] + reward[5])
print("reward 4: ", reward[1] + reward[3])
print("reward 5: ", reward[1] + reward[4])
print("reward 6: ", reward[1] + reward[5])
print("reward 7: ", reward[2] + reward[3])
print("reward 8: ", reward[2] + reward[4])
print("reward 9: ", reward[2] + reward[5])

{2: 256, 8: 249, 5: 345, 1: 57, 6: 42, 3: 67, 7: 73, 9: 60, 4: 43}
[-0.44658843  0.11995964  0.60712128 -0.88214324 -0.81949451 -0.75049923]


In [263]:
clusterDct[3].table[450:500]

,Unnamed: 0,bloc,icustayid,charttime,gender,age,elixhauser,re_admission,died_in_hosp,died_within_48h_of_out_time,...,output_total,output_4hourly,cumulated_balance,SOFA,SIRS,vaso_input,iv_input,discrete_states,actions,reward
1972,1972,1,643,4383658800,0,22033.667095,5,0,0,0,...,21235.0,650.0,7004.50,8,1,0.0,2.0,241,10,0.060400
1973,1973,2,643,4383673200,0,22033.667095,5,0,0,0,...,21915.0,680.0,6424.50,3,0,0.0,2.0,90,10,-0.094212
1974,1974,3,643,4383687600,0,22033.667095,5,0,0,0,...,22535.0,620.0,5904.50,8,0,0.0,2.0,242,10,0.072439
1975,1975,4,643,4383702000,0,22033.667095,5,0,0,0,...,23735.0,1200.0,4804.50,8,0,0.0,2.0,242,10,0.123856
1976,1976,5,643,4383716400,0,22033.667095,5,0,0,0,...,25155.0,1420.0,3424.50,7,0,0.0,1.0,212,5,0.060400
1977,1977,6,643,4383730800,0,22033.667095,5,0,0,0,...,26415.0,1260.0,2164.50,6,0,0.0,0.0,180,0,0.211667
1978,1978,7,643,4383745200,0,22033.667095,5,0,0,0,...,27625.0,1210.0,954.50,5,0,0.0,0.0,151,0,0.123856
1979,1979,8,643,4383759600,0,22033.667095,5,0,0,0,...,28925.0,1300.0,-245.50,4,0,0.0,2.0,121,10,0.072439
1980,1980,9,643,4383774000,0,22033.667095,5,0,0,0,...,30005.0,1080.0,-1075.50,4,1,0.0,3.0,121,15,0.072439
1981,1981,10,643,4383788400,0,22033.667095,5,0,0,0,...,30885.0,880.0,-1705.50,4,2,0.0,3.0,121,15,0.072439
